# 🌐 HTTP GET & FastAPI — Building a Patient Management API

**Goal:** understand what an API is, what the HTTP **GET** method does, and how to build read-only endpoints with **FastAPI**
(path parameters, query parameters, validation, error handling, sorting).

> This notebook follows the *Patient Management System* API written in the live class, and continues the [Pydantic notes](Pydantic_Notes.ipynb): FastAPI uses Pydantic internally for validation.

## 📑 Table of Contents
1. API, REST and HTTP basics
2. HTTP methods and the GET method
3. HTTP status codes
4. FastAPI: setup and first endpoints
5. The data: `patients.json`
6. `GET /view` — return all data
7. `GET /patient/{patient_id}` — **path parameters**, `Path()`, `HTTPException`
8. `GET /sort` — **query parameters**, `Query()`, validation, sorting
9. Bugs found in the live-class code (and fixes)
10. Improvement: `Literal` for automatic validation
11. Running the real server + Swagger docs
12. Final consolidated `main.py`
13. Summary, common mistakes, practice exercises

---
# 1️⃣ API, REST and HTTP basics

| Term | Meaning |
|------|---------|
| **API** (Application Programming Interface) | A set of rules that lets one program talk to another. |
| **Client** | The program that *asks* (browser, mobile app, Python script, another service). |
| **Server** | The program that *answers* (our FastAPI app). It holds the data / logic. |
| **HTTP** | The protocol (the "language") clients and servers use over the web. |
| **REST API** | An API design style where *resources* (patients) have URLs and we act on them using HTTP methods. |
| **Endpoint** | One URL + method combination, e.g. `GET /patient/P001`. |
| **JSON** | The text format used to send data back and forth. |

```
   Client  ──── HTTP Request  (method + URL + headers [+ body]) ────►  Server
   Client  ◄─── HTTP Response (status code + headers + body/JSON) ────  Server
```

### Anatomy of a URL
```
http://127.0.0.1:8000/sort?sort_by=bmi&order=desc
└─┬─┘   └────┬─────┘└┬─┘└┬─┘ └────────┬─────────┘
scheme     host    port path       query string
```
- **Path** (`/sort`, `/patient/P001`): *which resource* you want.
- **Query string** (`?sort_by=bmi&order=desc`): *extra options* (filter / sort / paginate), written as `key=value` pairs joined with `&`.

---
# 2️⃣ HTTP methods and the GET method

| Method | Purpose | CRUD | Has request body? |
|--------|---------|------|-------------------|
| **GET** | **Read** / fetch data | Read | No |
| POST | Create new data | Create | Yes |
| PUT | Replace / update existing data | Update | Yes |
| PATCH | Partially update | Update | Yes |
| DELETE | Remove data | Delete | Usually no |

## Properties of GET
- ✅ **Safe:** it only *reads*; it must **never change** data on the server.
- ✅ **Idempotent:** calling it 1 time or 100 times gives the same result and side effects.
- ✅ **Cacheable:** browsers/proxies can store responses.
- ❌ **No request body:** all inputs go in the **URL** (path parameters and query parameters).
- ⚠️ Because inputs live in the URL, **never send secrets (passwords, tokens) in a GET URL**; URLs get logged.

### Path parameter vs Query parameter
| | Path parameter | Query parameter |
|---|----------------|------------------|
| Looks like | `/patient/P001` | `/sort?sort_by=bmi&order=desc` |
| Purpose | **Identify** one specific resource | **Modify / filter / sort** how results are returned |
| Required? | Always required | Can be required *or* optional (with a default) |
| FastAPI tool | `Path()` | `Query()` |

---
# 3️⃣ HTTP status codes

The **status code** in every response tells the client what happened.

| Range | Meaning | Common codes |
|-------|---------|--------------|
| **2xx** | ✅ Success | `200 OK`, `201 Created`, `204 No Content` |
| **3xx** | ↪️ Redirect | `301 Moved Permanently` |
| **4xx** | ❌ *Client* made a mistake | `400 Bad Request`, `401 Unauthorized`, `403 Forbidden`, `404 Not Found`, `405 Method Not Allowed`, `422 Unprocessable Entity` |
| **5xx** | 💥 *Server* failed | `500 Internal Server Error` |

Codes we will use in this notebook:
- **200** → normal successful GET (FastAPI's default)
- **400** → the client sent an invalid *value* that we check manually (e.g. `sort_by=colour`)
- **404** → the requested patient does not exist
- **405** → wrong method (e.g. `POST` on a GET-only endpoint)
- **422** → FastAPI/Pydantic automatically rejects a request with a missing or wrongly typed parameter

---
# 4️⃣ FastAPI: setup and first endpoints

**FastAPI** is a modern Python web framework. Why it is popular:
- 🚀 Fast and asynchronous-ready
- ✅ Uses **type hints + Pydantic** for automatic validation
- 📖 Generates **interactive documentation** automatically (`/docs`)

### The building blocks
| Piece | Role |
|-------|------|
| `app = FastAPI()` | Creates the application object |
| `@app.get("/path")` | **Decorator** that registers the function below it as the handler for `GET /path` |
| function | Runs when a request arrives; whatever it **returns** (dict, list, ...) is converted to JSON automatically |

### Install
```bash
pip install fastapi uvicorn httpx2
```
`uvicorn` is the server that runs the app. `httpx2` is needed for the test client used in this notebook (on older Starlette/FastAPI versions install `httpx` instead).

### How we test inside a notebook
Normally you run the server in a terminal (section 11). Inside Jupyter it is easier to use FastAPI's **`TestClient`**, which sends
requests to the app **without starting a real server**. It behaves like a browser/`requests` and returns real status codes and JSON.

In [1]:
# %pip install fastapi uvicorn httpx2     # uncomment if not installed (older versions: httpx)

import json
from fastapi import FastAPI, Path, HTTPException, Query
from fastapi.testclient import TestClient

app = FastAPI()               # create the application
client = TestClient(app)      # a fake browser to call our endpoints from the notebook


def show(response):
    """Helper: print the status code and the JSON body neatly."""
    print("Status code:", response.status_code)
    print(json.dumps(response.json(), indent=2))

## First two endpoints: `/` and `/about`
The simplest GET endpoints: no parameters, they just return a dictionary.

In [2]:
@app.get("/")
def hello():
    # Home route: a quick health check that the API is alive
    return {'message': 'Patient Management System API'}


@app.get("/about")
def about():
    # Describes what the API does
    return {'message': 'A fully functional API to manage your patient data'}

In [3]:
show(client.get("/"))

Status code: 200
{
  "message": "Patient Management System API"
}


In [4]:
show(client.get("/about"))

Status code: 200
{
  "message": "A fully functional API to manage your patient data"
}


> 💡 A returned Python `dict` is automatically serialized to JSON by FastAPI; you never call `json.dumps` yourself.

---
# 5️⃣ The data: `patients.json`

The API reads from a JSON file that acts as a tiny database. In the live class this file was given; here we **create sample data ourselves** so the notebook is self-contained.

Structure: a dictionary where the **key is the patient id** and the value holds the details.
```json
{ "P001": {"name": "...", "city": "...", "age": 28, "gender": "Female", "height": 1.65, "weight": 90, "bmi": 33.06, "verdict": "Obese"} }
```

In [5]:
# Build sample data. BMI = weight(kg) / height(m)^2  (same formula as our Pydantic computed_field)
def bmi_verdict(bmi):
    if bmi < 18.5:  return 'Underweight'
    if bmi < 25:    return 'Normal'
    if bmi < 30:    return 'Overweight'
    return 'Obese'

raw = [
    # id,    name,            city,        age, gender,  height(m), weight(kg)
    ('P001', 'Ananya Verma',   'Delhi',     28, 'Female', 1.65, 90),
    ('P002', 'Rohan Mehta',    'Mumbai',    35, 'Male',   1.80, 72),
    ('P003', 'Sita Sharma',    'Kathmandu', 42, 'Female', 1.58, 48),
    ('P004', 'Aman Gupta',     'Bengaluru', 61, 'Male',   1.72, 85),
    ('P005', 'Priya Nair',     'Chennai',   30, 'Female', 1.68, 60),
]

patients = {}
for pid, name, city, age, gender, height, weight in raw:
    bmi = round(weight / height ** 2, 2)
    patients[pid] = {'name': name, 'city': city, 'age': age, 'gender': gender,
                     'height': height, 'weight': weight, 'bmi': bmi, 'verdict': bmi_verdict(bmi)}

# Save it as patients.json (our "database")
with open('patients.json', 'w') as f:
    json.dump(patients, f, indent=2)

print(json.dumps(patients['P001'], indent=2))

{
  "name": "Ananya Verma",
  "city": "Delhi",
  "age": 28,
  "gender": "Female",
  "height": 1.65,
  "weight": 90,
  "bmi": 33.06,
  "verdict": "Obese"
}


## Helper: `load_data()`
Reads the file and returns it as a Python dictionary.

- `with open(...)` is a **context manager**: it closes the file automatically, even if an error occurs.
- `'r'` = read mode. `json.load(f)` converts JSON text → Python `dict`.
- Every request reloads the file, so changes to the file show up immediately. Fine for learning; a real app would use a database.

In [6]:
def load_data():
    with open('patients.json', 'r') as f:
        data = json.load(f)
    return data

len(load_data())

5

---
# 6️⃣ `GET /view` — return all patients

The simplest data endpoint: load the data and return it.

In [7]:
@app.get('/view')
def view():
    data = load_data()
    return data           # dict → JSON automatically

In [8]:
response = client.get('/view')
print("Status:", response.status_code)
print("Number of patients:", len(response.json()))
print("Patient ids:", list(response.json().keys()))

Status: 200
Number of patients: 5
Patient ids: ['P001', 'P002', 'P003', 'P004', 'P005']


### Methods are enforced
`/view` is registered **only for GET**. Other methods are rejected with **405 Method Not Allowed**.

In [9]:
show(client.post('/view'))

Status code: 405
{
  "detail": "Method Not Allowed"
}


---
# 7️⃣ `GET /patient/{patient_id}` — Path parameters

**Goal:** fetch **one** patient by id, e.g. `/patient/P002`.

### Concepts
- `{patient_id}` in the route is a **path parameter**: a placeholder filled from the URL.
- The function must have an argument with the **same name** (`patient_id`).
- The type hint (`str`) tells FastAPI/Pydantic how to validate and convert it.
- **`Path()`** adds validation and documentation metadata to a path parameter.

### `Path(...)`: what does `...` mean?
The **Ellipsis** `...` means *"this has no default, it is required"*. Path parameters are always required, so we write `Path(...)`.

### Error handling with `HTTPException`
If the id is not in our data we must tell the client, not return `None` or crash.
`raise HTTPException(status_code=404, detail='...')` stops the function and sends a proper JSON error response:
```json
{"detail": "Patient not found"}
```

In [10]:
@app.get('/patient/{patient_id}')
def view_patient(
    patient_id: str = Path(
        ...,                                             # required
        description='ID of the patient in the DB',       # shown in /docs
        examples=['P001'],                               # example value shown in /docs
    )
):
    data = load_data()

    if patient_id in data:          # dictionary key lookup
        return data[patient_id]

    # Not found → 404 (client asked for something that does not exist)
    raise HTTPException(status_code=404, detail='Patient not found')

In [11]:
# ✅ Existing patient → 200
show(client.get('/patient/P002'))

Status code: 200
{
  "name": "Rohan Mehta",
  "city": "Mumbai",
  "age": 35,
  "gender": "Male",
  "height": 1.8,
  "weight": 72,
  "bmi": 22.22,
  "verdict": "Normal"
}


In [12]:
# ❌ Unknown patient → 404
show(client.get('/patient/P999'))

Status code: 404
{
  "detail": "Patient not found"
}


### Where does the metadata go?
`description` and `examples` do not change behaviour; they are used in the auto-generated **OpenAPI schema** (which powers `/docs`).

In [13]:
# Peek at the auto-generated documentation for this endpoint
print(json.dumps(app.openapi()['paths']['/patient/{patient_id}']['get']['parameters'], indent=2))

[
  {
    "name": "patient_id",
    "in": "path",
    "required": true,
    "schema": {
      "type": "string",
      "description": "ID of the patient in the DB",
      "examples": [
        "P001"
      ],
      "title": "Patient Id"
    },
    "description": "ID of the patient in the DB"
  }
]


> ⚠️ **Note:** the live-class code used `example='P001'`. In current FastAPI versions `example=` is **deprecated**; use `examples=['P001']`.

> ⚠️ **Route order matters:** FastAPI matches routes top-to-bottom. A fixed route like `/patient/all` must be declared **before** `/patient/{patient_id}`, otherwise `{patient_id}` would swallow `"all"`.

---
# 8️⃣ `GET /sort` — Query parameters

**Goal:** return all patients **sorted** by `height`, `weight` or `bmi`, in ascending or descending order:
```
GET /sort?sort_by=bmi&order=desc
```

### Concepts
- Any function argument that is **not** in the route path becomes a **query parameter**.
- **`Query()`** adds validation and metadata, just like `Path()` does for path parameters.
- `Query(...)` → **required**. `Query('asc')` → **optional, default is `'asc'`**.
- FastAPI returns **422** automatically when a required query parameter is missing.
- We do **manual validation** for business rules (allowed field names, allowed order) and raise **400 Bad Request**.

### How sorting works in Python
```python
sorted(iterable, key=function, reverse=True/False)
```
- `data.values()` → the patient dictionaries (we drop the ids; see the note below).
- `key=lambda x: x.get(sort_by, 0)`: for each patient `x`, use the value of the chosen field for comparison. `.get(field, 0)` avoids a `KeyError` if a field is missing.
- `reverse=True` → descending, `reverse=False` → ascending.

In [14]:
@app.get('/sort')
def sort_patients(
    sort_by: str = Query(..., description='Sort on the basis of height, weight or bmi'),   # required
    order: str = Query('asc', description='Sort in asc or desc order'),                    # optional (default 'asc')
):
    # 1) Validate the inputs (business rules) → 400 Bad Request if wrong
    valid_fields = ['height', 'weight', 'bmi']
    if sort_by not in valid_fields:
        raise HTTPException(status_code=400, detail=f'Invalid field, select from {valid_fields}')

    if order not in ['asc', 'desc']:
        raise HTTPException(status_code=400, detail='Invalid order, select either asc or desc')

    # 2) Convert the order text into the boolean that sorted() understands
    sort_order = True if order == 'desc' else False      # True → reverse (descending)

    # 3) Load and sort
    data = load_data()
    sorted_data = sorted(data.values(), key=lambda x: x.get(sort_by, 0), reverse=sort_order)

    return sorted_data

In [15]:
# Ascending BMI (order not given → default 'asc')
result = client.get('/sort', params={'sort_by': 'bmi'})
print("Status:", result.status_code)
for p in result.json():
    print(f"{p['name']:<15} BMI={p['bmi']}")

Status: 200
Sita Sharma     BMI=19.23
Priya Nair      BMI=21.26
Rohan Mehta     BMI=22.22
Aman Gupta      BMI=28.73
Ananya Verma    BMI=33.06


In [16]:
# Descending weight
result = client.get('/sort?sort_by=weight&order=desc')          # same as params={...}
print("Status:", result.status_code)
for p in result.json():
    print(f"{p['name']:<15} weight={p['weight']}")

Status: 200
Ananya Verma    weight=90
Aman Gupta      weight=85
Rohan Mehta     weight=72
Priya Nair      weight=60
Sita Sharma     weight=48


### The three kinds of errors, side by side

In [17]:
# 400 → our manual validation: invalid sort field
show(client.get('/sort', params={'sort_by': 'colour'}))

Status code: 400
{
  "detail": "Invalid field, select from ['height', 'weight', 'bmi']"
}


In [18]:
# 400 → our manual validation: invalid order
show(client.get('/sort', params={'sort_by': 'bmi', 'order': 'sideways'}))

Status code: 400
{
  "detail": "Invalid order, select either asc or desc"
}


In [19]:
# 422 → FastAPI's automatic validation: required query param 'sort_by' is missing
show(client.get('/sort'))

Status code: 422
{
  "detail": [
    {
      "type": "missing",
      "loc": [
        "query",
        "sort_by"
      ],
      "msg": "Field required",
      "input": null
    }
  ]
}


> 📝 **Design note:** `sorted(data.values(), ...)` returns a **list** of patients *without* their ids (`P001`, ...). If the client needs the id, include it inside each record, or sort `data.items()` instead.

---
# 9️⃣ Bugs found in the live-class code (and fixes)

While converting the class code into these notes, three real bugs/issues appeared. Good reading practice: none of them crash the app, they just silently give **wrong behaviour**.

| # | Live-class code | Problem | Fix |
|---|-----------------|---------|-----|
| 1 | `valid_fields = ['heights', 'weight', 'bmi']` | Typo. The data key is `height` (singular), so `sort_by=height` is rejected with 400 and `sort_by=heights` passes validation but every `x.get('heights', 0)` returns `0` → **no real sorting**. | `['height', 'weight', 'bmi']` |
| 2 | `sorted(..., reverse=False)` | `sort_order` is computed but **never used**, so `order=desc` still returns ascending results. | `reverse=sort_order` |
| 3 | `Path(..., example='P001')` | `example=` is deprecated in current FastAPI. | `examples=['P001']` |

Let's *prove* bug #2 with plain Python (no API needed):

In [20]:
data = load_data()
order = 'desc'
sort_order = True if order == 'desc' else False

# ❌ Buggy: reverse is hard-coded to False, so 'desc' is ignored
buggy = sorted(data.values(), key=lambda x: x.get('weight', 0), reverse=False)

# ✅ Fixed: use the computed boolean
fixed = sorted(data.values(), key=lambda x: x.get('weight', 0), reverse=sort_order)

print("order requested: desc")
print("buggy :", [p['weight'] for p in buggy], "← ascending, wrong!")
print("fixed :", [p['weight'] for p in fixed], "← descending, correct")

order requested: desc
buggy : [48, 60, 72, 85, 90] ← ascending, wrong!
fixed : [90, 85, 72, 60, 48] ← descending, correct


In [21]:
# And bug #1: the typo makes every value default to 0, so the 'sort' does nothing useful
typo_sorted = sorted(data.values(), key=lambda x: x.get('heights', 0))   # 'heights' does not exist
print("keys used for comparison:", [x.get('heights', 0) for x in typo_sorted])
print("→ all zeros, so the original order is kept (no sorting happened)")

keys used for comparison: [0, 0, 0, 0, 0]
→ all zeros, so the original order is kept (no sorting happened)


---
# 🔟 Improvement: `Literal` for automatic validation

Instead of validating `sort_by` and `order` by hand with `if` statements, we can declare the **allowed values in the type** using `typing.Literal`.

Benefits:
- ✅ FastAPI/Pydantic validates automatically (**422** error listing allowed values)
- ✅ Less code
- ✅ Swagger UI shows a **dropdown** with the allowed choices
- ✅ Editors and type checkers understand the allowed values

This is the same idea as Pydantic validation in the previous notebook, now applied to URL parameters.

In [22]:
from typing import Literal

@app.get('/sort-v2')
def sort_patients_v2(
    sort_by: Literal['height', 'weight', 'bmi'] = Query(..., description='Field to sort by'),
    order: Literal['asc', 'desc'] = Query('asc', description='Sort order'),
):
    data = load_data()
    return sorted(data.values(), key=lambda x: x.get(sort_by, 0), reverse=(order == 'desc'))

In [23]:
# Valid request
result = client.get('/sort-v2', params={'sort_by': 'height', 'order': 'desc'})
print("Status:", result.status_code, "|", [p['height'] for p in result.json()])

Status: 200 | [1.8, 1.72, 1.68, 1.65, 1.58]


In [24]:
# Invalid value → FastAPI returns 422 and even lists the allowed choices
show(client.get('/sort-v2', params={'sort_by': 'colour'}))

Status code: 422
{
  "detail": [
    {
      "type": "literal_error",
      "loc": [
        "query",
        "sort_by"
      ],
      "msg": "Input should be 'height', 'weight' or 'bmi'",
      "input": "colour",
      "ctx": {
        "expected": "'height', 'weight' or 'bmi'"
      }
    }
  ]
}


---
# 1️⃣1️⃣ Running the real server + Swagger docs

`TestClient` is perfect for notebooks/tests, but a real API runs as a server. Save the code as `main.py` (see section 12) and run in a **terminal**:

```bash
uvicorn main:app --reload
```
| Part | Meaning |
|------|---------|
| `main` | the file name `main.py` |
| `app` | the variable `app = FastAPI()` inside that file |
| `--reload` | restart automatically when you edit the code (development only) |

Then open in the browser:

| URL | What you get |
|-----|--------------|
| `http://127.0.0.1:8000/view` | Your endpoint's JSON |
| `http://127.0.0.1:8000/docs` | **Swagger UI**: interactive docs where you can *try* every endpoint |
| `http://127.0.0.1:8000/redoc` | ReDoc: alternative documentation |
| `http://127.0.0.1:8000/openapi.json` | The raw OpenAPI schema |

> 💡 Because `Path()`/`Query()` descriptions and examples feed the docs, well-described parameters give you free, professional documentation.

You can also call the running server from Python with `requests`:
```python
import requests
requests.get('http://127.0.0.1:8000/sort', params={'sort_by': 'bmi', 'order': 'desc'}).json()
```

---
# 1️⃣2️⃣ Final consolidated `main.py` (with all fixes)

The cell below writes the complete, corrected application to a file using the `%%writefile` magic, ready for `uvicorn main:app --reload`.

In [25]:
%%writefile main.py
from fastapi import FastAPI, Path, HTTPException, Query
import json

app = FastAPI()


def load_data():
    """Load all patient records from the JSON file (our simple database)."""
    with open('patients.json', 'r') as f:
        return json.load(f)


@app.get("/")
def hello():
    return {'message': 'Patient Management System API'}


@app.get("/about")
def about():
    return {'message': 'A fully functional API to manage your patient data'}


@app.get('/view')
def view():
    """Return every patient."""
    return load_data()


@app.get('/patient/{patient_id}')
def view_patient(
    patient_id: str = Path(..., description='ID of the patient in the DB', examples=['P001'])
):
    """Return a single patient by id, or 404 if it does not exist."""
    data = load_data()
    if patient_id in data:
        return data[patient_id]
    raise HTTPException(status_code=404, detail='Patient not found')


@app.get('/sort')
def sort_patients(
    sort_by: str = Query(..., description='Sort on the basis of height, weight or bmi'),
    order: str = Query('asc', description='Sort in asc or desc order'),
):
    """Return all patients sorted by height, weight or bmi."""
    valid_fields = ['height', 'weight', 'bmi']          # fixed typo: 'heights' -> 'height'

    if sort_by not in valid_fields:
        raise HTTPException(status_code=400, detail=f'Invalid field, select from {valid_fields}')
    if order not in ['asc', 'desc']:
        raise HTTPException(status_code=400, detail='Invalid order, select either asc or desc')

    sort_order = True if order == 'desc' else False     # True -> descending
    data = load_data()


    # fixed: use reverse=sort_order (was hard-coded to False)
    return sorted(data.values(), key=lambda x: x.get(sort_by, 0), reverse=sort_order)

Writing main.py


---
# 1️⃣3️⃣ Summary

## 🧠 Cheat-sheet
| Concept | Code | Notes |
|---------|------|-------|
| Create app | `app = FastAPI()` | one per project |
| GET endpoint | `@app.get('/route')` | decorator + function |
| Return data | `return dict_or_list` | auto-converted to JSON |
| Path parameter | `'/patient/{patient_id}'` + `patient_id: str = Path(...)` | identifies **one** resource, always required |
| Query parameter | `sort_by: str = Query(...)` | filter/sort options; `Query('asc')` = optional with default |
| Required marker | `...` (Ellipsis) | "no default, must be supplied" |
| Error response | `raise HTTPException(status_code=404, detail='...')` | proper HTTP error + JSON message |
| Auto validation | type hints, `Literal[...]` | invalid input → **422** |
| Run server | `uvicorn main:app --reload` | docs at `/docs` |

## 🔑 Key ideas
1. **GET = read-only.** It is safe and idempotent; inputs travel in the URL.
2. **Path param** identifies *which* resource; **query param** controls *how* the list is returned.
3. Use the **right status code**: 200 OK, 400 bad value, 404 not found, 422 validation, 500 server bug.
4. FastAPI + Pydantic give **validation and documentation for free**.

## ⚠️ Common mistakes
1. Typos in allowed field names (`heights` vs `height`). Better: use `Literal`.
2. Computing `sort_order` but hard-coding `reverse=False`.
3. Returning `None` / an error message with status 200 instead of raising `HTTPException`.
4. Putting a fixed route (`/patient/all`) *after* a dynamic one (`/patient/{id}`).
5. Using `example=` (deprecated) instead of `examples=[...]`.
6. Trying to *change* data with GET. Use POST/PUT/PATCH/DELETE for that.
7. Running `uvicorn` from a different folder, so `patients.json` isn't found (relative path).

## 🎯 Practice exercises
1. Add `GET /patients/city/{city}` that returns all patients from a city (case-insensitive).
2. Add a query parameter `limit: int = Query(3, ge=1, le=10)` to `/sort` to return only the top *N* patients.
3. Add `GET /filter?min_age=30&max_age=50` with optional query parameters (`Optional[int] = None`).
4. Make `/sort` include the patient id inside every returned record.
5. Add `GET /stats` returning the average BMI and the count of each `verdict`.
6. Replace `load_data()`'s JSON file with a Pydantic model list and return `response_model=List[Patient]`.

## 🔗 References
- FastAPI docs: <https://fastapi.tiangolo.com/>
- Path parameters: <https://fastapi.tiangolo.com/tutorial/path-params/>
- Query parameters: <https://fastapi.tiangolo.com/tutorial/query-params/>
- MDN: HTTP request methods: <https://developer.mozilla.org/en-US/docs/Web/HTTP/Methods>